# GPT-OSS 120B — Nemotron Low Reasoning SFT Training

> **Multi-run SFT** on the Nemotron "low reasoning pass-rate" dataset using Unsloth + LoRA.  
> The 50k dataset is split across multiple runs (each limited to 1500 steps) to stay within Kaggle's timeout.

| Parameter | Value |
|---|---|
| Base Model | `gpt-oss-120b-bnb-4bit` |
| Method | LoRA (r=32, α=64) |
| Quantization | 4-bit (BnB) |
| Effective Batch Size | 2 (1 × 2 grad accum) |
| Samples per Run | 3,000 |
| Total Runs | ~17 |

---
## 1 · Configuration

Set `Training_RUN` to the current run index (0, 1, 2, …).  
The dataset is automatically sliced so each run trains on a **non-overlapping** portion.

In [ ]:
class CONFIG:
    # ── Training Hyperparameters ──
    TRAIN_SIZE              = 50_000
    BATCH_SIZE              = 1
    GRADIENT_ACCUMULATION   = 2
    MAX_STEPS_PER_RUN       = 1500
    LEARNING_RATE           = 1e-4
    MAX_SEQ_LENGTH          = 8_000

    # ── Environment ──
    KAGGLE                  = True
    MASK_THINK              = False

    # ── Multi-Run Control ──────────────────────────────
    #  Change this for each successive run: 0, 1, 2, …
    Training_RUN            = 0
    # ───────────────────────────────────────────────────

cfg = CONFIG()

# ── Derived Paths ──
cfg.MODEL_PATH = (
    "/kaggle/input/models/barnobarno/gpt-oss-120b-bnb-4bit/transformers/unsloth/1"
    if cfg.KAGGLE else "unsloth/gpt-oss-20b"
)
cfg.LORA_PATH = (
    "/kaggle/input/models/barnobarno/gpt-oss-120b-bnb-4bit/transformers/unsloth/1/lora"
    if cfg.KAGGLE else "unsloth/gpt-oss-20b-lora"
)

# ── Multi-Run Calculations ──
cfg.EFFECTIVE_BATCH_SIZE = cfg.BATCH_SIZE * cfg.GRADIENT_ACCUMULATION
cfg.SAMPLES_PER_RUN      = cfg.MAX_STEPS_PER_RUN * cfg.EFFECTIVE_BATCH_SIZE
cfg.TOTAL_RUNS_NEEDED    = -(-cfg.TRAIN_SIZE // cfg.SAMPLES_PER_RUN)  # ceil division

print("=" * 50)
print("  MULTI-RUN TRAINING CONFIGURATION")
print("=" * 50)
print(f"  Effective batch size   : {cfg.EFFECTIVE_BATCH_SIZE}")
print(f"  Samples per run        : {cfg.SAMPLES_PER_RUN}")
print(f"  Total runs to cover {cfg.TRAIN_SIZE:,} samples : {cfg.TOTAL_RUNS_NEEDED}")
print(f"  ▶ Current run          : {cfg.Training_RUN} / {cfg.TOTAL_RUNS_NEEDED - 1}")
print("=" * 50)

---
## 2 · Environment Setup

Install Unsloth and dependencies. The first cell is for non-Kaggle environments (commented out).  
The second cell installs from a local Kaggle dataset for offline runs.

In [ ]:
if cfg.KAGGLE:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-py-3-12/unsloth' 'unsloth'


In [ ]:
try:
    from unsloth import FastLanguageModel
except:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

---
## 3 · Model Loading

Load the base GPT-OSS 120B model in 4-bit quantization.  
The LoRA adapter is attached separately in **Section 6**.

In [ ]:
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.MODEL_PATH,
    dtype=None,                     # Auto-detect
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    local_files_only=cfg.KAGGLE,    # True on Kaggle (offline)
)
print(f"Model loaded from: {cfg.MODEL_PATH}")

---
## 4 · Data Loading

Load the Nemotron low-reasoning pass-rate dataset (filtered to pass rate 1 or 2).

In [ ]:
import pandas as pd
if not cfg.KAGGLE:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    # Set the path to the file you'd like to load
    file_path = "filtered_low_pass_1_or_2.jsonl"

    # Load the latest version
    df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "barnobarno/nemotron-low-reasoning-pass-rate-1-2",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
    )
if cfg.KAGGLE:
    df = pd.read_json("/kaggle/input/nemotron-low-reasoning-pass-rate-1-2/filtered_low_pass_1_or_2.jsonl", lines=True)
print("DONE DATA LOADING")


---
## 5 · Data Preprocessing & Multi-Run Slicing

1. Filter out tool-call rows and reformat messages for GPT-OSS template  
2. Deduplicate by `expected_answer` and sample `TRAIN_SIZE` rows (deterministic seed)  
3. **Slice** the dataset for the current `Training_RUN` so each run sees fresh, non-overlapping data  
4. Apply the chat template

In [ ]:
from datasets import Dataset

# ── Helper Functions ──

def remove_none_keys(messages):
    """Strip None-valued keys from each message dict."""
    return [{k: v for k, v in entry.items() if v is not None} for entry in messages]

def format_for_gpt_oss(example):
    """Rename 'reasoning_content' → 'thinking' and sanitise tool-call messages."""
    messages = example['messages']
    new_messages = []
    for msg in messages:
        new_msg = msg.copy()
        if 'reasoning_content' in new_msg:
            new_msg['thinking'] = new_msg.pop('reasoning_content')
        if new_msg.get('tool_calls') and new_msg.get('thinking'):
            new_msg['content'] = ""
        new_messages.append(new_msg)
    return {'messages': new_messages}

# ── Clean & Format ──

train_data = df.copy()
train_data.drop(
    columns=["uuid", "original_expected_answer", "license",
             "used_in", "user_name", "user_url", "url"],
    inplace=True,
)
train_data = train_data[train_data["tools"].isna()]
train_data.drop(columns=["tools"], inplace=True)

train_data["messages"] = train_data.apply(format_for_gpt_oss, axis=1)
train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])

# ── Deduplicate + Sample Full Pool (deterministic) ──

unique_pool = train_data.drop_duplicates(subset=["expected_answer"])
full_dataset = unique_pool.sample(n=cfg.TRAIN_SIZE, random_state=42).reset_index(drop=True)
print(f"Full dataset pool : {len(full_dataset):,} samples")

# ── Slice for Current Training Run ──

start_idx = cfg.Training_RUN * cfg.SAMPLES_PER_RUN
end_idx   = min((cfg.Training_RUN + 1) * cfg.SAMPLES_PER_RUN, len(full_dataset))

assert start_idx < len(full_dataset), (
    f"Training_RUN {cfg.Training_RUN} is out of range! "
    f"Only {cfg.TOTAL_RUNS_NEEDED} runs needed."
)

dataset = full_dataset.iloc[start_idx:end_idx].copy()
print(f"Run {cfg.Training_RUN}       : samples [{start_idx} : {end_idx}]  →  {len(dataset):,} rows")

# ── Apply Chat Template ──

dataset["AA"] = dataset["AA"].apply(remove_none_keys)
dataset["text"] = dataset.apply(
    lambda row: tokenizer.apply_chat_template(
        row["AA"],
        tokenize=False,
        add_generation_prompt=False,
        reasoning_effort="low",
    ),
    axis=1,
)

print(f"Unique answers    : {dataset['expected_answer'].nunique()}")
print(f"Dataset ready     : {dataset.shape}")

---
## 6 · LoRA Adapter Setup

- **Run 0**: Create a fresh LoRA adapter (r=32, α=64)  
- **Run ≥ 1**: Load the LoRA checkpoint saved from the previous run

In [ ]:
import gc

hf_dataset = Dataset.from_pandas(dataset)

if cfg.Training_RUN == 0:
    # First run: create fresh LoRA adapter
    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    print("Created fresh LoRA adapter (r=32, α=64)")
else:
    # Subsequent runs: load LoRA from previous checkpoint
    model = model.load_adapter(cfg.LORA_PATH)
    print(f"Loaded LoRA adapter from: {cfg.LORA_PATH}")

gc.collect()
torch.cuda.empty_cache()

---
## 7 · Trainer Configuration

Build the SFTTrainer and apply response-only masking so the loss is computed  
only on the assistant's output (analysis or final channel, controlled by `MASK_THINK`).

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION,
        warmup_steps=5,
        max_steps=cfg.MAX_STEPS_PER_RUN,
        learning_rate=cfg.LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=666,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        group_by_length=True,
    ),
)
print(f"Trainer built — max_steps={cfg.MAX_STEPS_PER_RUN}, lr={cfg.LEARNING_RATE}")

In [ ]:
# CHANGE THIS: Point to the analysis channel instead of the final channel sometime
if not cfg.MASK_THINK:
    try:
        gpt_oss_kwargs = dict(
            instruction_part = "<|start|>user<|message|>", 
            response_part = "<|start|>assistant<|channel|>analysis<|message|>"
        )
        
        trainer = train_on_responses_only(
            trainer,
            **gpt_oss_kwargs,
        )
        TRAIN=True
    except:
        TRAIN = False
else:
    gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)
    trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)
    TRAIN = True



    


print(f"TRAINING READY AS NOT MASKING THE THINKING FROM TRAIN LOSS {cfg.MASK_THINK}")

---
## 8 · Training

Run training for `MAX_STEPS_PER_RUN` steps on the current data slice.

In [ ]:
trainer_stats = trainer.train()

# ── Stats ──
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
train_time_seconds = trainer_stats.metrics.get('train_runtime', 0)
train_time_minutes = round(train_time_seconds / 60, 2)

print(f"\n{'=' * 50}")
print(f"  Run {cfg.Training_RUN} Complete")
print(f"{'=' * 50}")
print(f"  Peak reserved memory : {used_memory} GB")
print(f"  Training time        : {train_time_minutes} min ({train_time_seconds:.1f}s)")
print(f"  Final loss           : {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"{'=' * 50}")

---
## 9 · Save Checkpoint

Save the LoRA adapter for this run. Use the output path as `LORA_PATH` in the next run.

In [ ]:
save_name = f"gpt_oss_120b_nemotron_low_run{cfg.Training_RUN}"

model.save_pretrained(save_name)
tokenizer.save_pretrained(save_name)

print(f"Checkpoint saved to: '{save_name}/'")
print(f"For the next run, set:")
print(f"  cfg.Training_RUN = {cfg.Training_RUN + 1}")
print(f"  cfg.LORA_PATH    = '{save_name}'")